# Session 2: how hard does the soil push?
**A. Real-world problem.** How big is the sideways push of 4 m of soil on a wall, where
does it act, and what happens when a car parks behind the wall or the ground fills with water?

**Plan.** (1) the coefficient Ka in two forms, (2) the triangular pressure and its resultant,
(3) a surcharge, (4) water, (5) a design question, (6) how much Ka depends on phi.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

GAMMA_W = 9.81     # unit weight of water, kN/m^3

## Step 1: the active coefficient Ka
For a smooth vertical wall and level backfill, with soil friction angle phi:
    Ka = tan^2(45 - phi/2) = (1 - sin phi) / (1 + sin phi).
For phi = 30 degrees: sin 30 = 0.5, so Ka = 0.5 / 1.5 = 1/3.

In [ ]:
def ka_tan(phi_deg):
    return float(np.tan(np.radians(45.0 - phi_deg / 2.0)) ** 2)


def ka_sin(phi_deg):
    s = np.sin(np.radians(phi_deg))
    return float((1.0 - s) / (1.0 + s))

## Step 2: pressure, resultant, arm
Horizontal pressure at depth z is p = Ka * gamma * z (kPa = kN per square metre). It is a
triangle from 0 at the top to Ka gamma H at the base. The area of the triangle is the force per
metre of wall: Pa = 0.5 Ka gamma H^2, acting H/3 above the base.

In [ ]:
def pressure(z, phi, gamma):
    """Horizontal pressure (kPa) at depth z (m) below the surface."""
    return ka_sin(phi) * gamma * np.asarray(z, dtype=float)


def active_force(H, phi, gamma):
    """Resultant Pa in kN per metre run."""
    return 0.5 * ka_sin(phi) * gamma * H ** 2


def surcharge_force(H, phi, q):
    """Extra force Ka q H (kN/m) from a uniform surface load q (kPa); it acts at H/2."""
    return ka_sin(phi) * q * H


def max_height(Pa_limit, phi, gamma):
    """Inverse problem: tallest wall whose soil force stays within Pa_limit (kN/m)."""
    return float(np.sqrt(2.0 * Pa_limit / (ka_sin(phi) * gamma)))

## Step 3: water, the warning
Water pushes with the full hydrostatic pressure gamma_w * z whatever the soil. For a
backfill that is saturated up to the top, the soil's effective weight drops to
gamma' = gamma_sat - gamma_w, and the water adds 0.5 gamma_w H^2.

In [ ]:
def saturated_force(H, phi, gamma_sat):
    """Soil (submerged) plus water force, kN/m, for backfill saturated to the top."""
    soil = 0.5 * ka_sin(phi) * (gamma_sat - GAMMA_W) * H ** 2
    water = 0.5 * GAMMA_W * H ** 2
    return soil, water, soil + water

## Step 4: see it
Left: the pressure triangle and its resultant for H = 4 m. Right: Ka against phi, which
falls steeply: a small error in phi is a big error in the push.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    H, phi, gamma = 4.0, 30.0, 18.0
    print(f"Ka tan form = {ka_tan(30):.4f}, sin form = {ka_sin(30):.4f}")
    print(f"pressure at the base = {float(pressure(H, phi, gamma)):.1f} kPa, Pa = {active_force(H, phi, gamma):.1f} kN/m at {H / 3:.2f} m")
    print(f"surcharge 10 kPa adds {surcharge_force(H, phi, 10):.1f} kN/m")
    so, wa, tot = saturated_force(H, phi, 20.0)
    print(f"saturated: soil {so:.1f} + water {wa:.1f} = {tot:.1f} kN/m  (dry: {active_force(H, phi, gamma):.1f})")
    print(f"phi 25: Ka {ka_sin(25):.4f} Pa {active_force(H, 25, gamma):.1f};  phi 35: Ka {ka_sin(35):.4f} Pa {active_force(H, 35, gamma):.1f}")
    print(f"tallest wall for 60 kN/m: {max_height(60, phi, gamma):.2f} m")

    z = np.linspace(0, H, 50)
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.fill_betweenx(z, 0, pressure(z, phi, gamma), alpha=0.4)
    ax.plot(pressure(z, phi, gamma), z)
    ax.axhline(H - H / 3, color="crimson", ls="--")
    ax.text(1, H - H / 3 - 0.2, "Pa acts at H/3 above the base", color="crimson")
    ax.invert_yaxis()
    ax.set_xlabel("horizontal pressure p (kPa)")
    ax.set_ylabel("depth below the surface z (m)")
    ax.set_title("Triangular pressure: p = Ka gamma z")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "pressure_triangle.png", dpi=120)

    ph = np.linspace(15, 45, 61)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ph, [ka_sin(p) for p in ph])
    ax2.plot([25, 35], [ka_sin(25), ka_sin(35)], "o")
    ax2.set_xlabel("soil friction angle phi (degrees)")
    ax2.set_ylabel("active coefficient Ka (no unit)")
    ax2.set_title("Ka falls steeply with phi")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "ka_vs_phi.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `ka_tan(p)` and `ka_sin(p)` for phi = 20, 30 and 40 degrees and show they agree.
2. **Stretch.** Add a function `total_force(H, phi, gamma, q)` that returns the soil plus surcharge force and the height at which it acts, and check it against the worked example (61.33 kN/m at 1.48 m).
3. **Explain.** Explain why a heavy rain can be more dangerous for a retaining wall than a heavy truck parked behind it.

## Assessment
- **A. Conceptual.** Why is the earth pressure on a wall a triangle, and why does the resultant act at H/3 above the base rather than at mid-height?
- **B. Numerical.** A wall 3 m high holds soil with gamma = 19 kN/m^3 and phi = 30 degrees. Find Pa and the height at which it acts.
- **C. Graph and simulation interpretation.** From the graph of Ka against phi, what happens to Ka when phi rises from 25 to 35 degrees? What does this mean for the 4 m wall's push?
- **D. Design and optimisation challenge.** Design: the wall may carry at most 60 kN/m of soil force. Find the tallest wall for gamma = 18 kN/m^3 and phi = 30 degrees, first without and then with a 10 kPa surcharge. Which is the stronger limit?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    phis = np.linspace(5, 60, 12)
    checks.append(("the tan^2(45 - phi/2) and (1 - sin)/(1 + sin) forms agree for every phi",
                   all(abs(L.ka_tan(p) - L.ka_sin(p)) < 1e-12 for p in phis)))
    checks.append(("phi = 30 degrees gives Ka = 1/3", abs(L.ka_sin(30.0) - 1.0 / 3.0) < 1e-12))
    checks.append(("edge: phi = 0 (a fluid) gives Ka = 1", abs(L.ka_sin(0.0) - 1.0) < 1e-12))
    checks.append(("pressure at the base of 4 m is Ka gamma H = 24 kPa and zero at the top",
                   abs(float(L.pressure(4.0, 30.0, 18.0)) - 24.0) < 1e-9 and float(L.pressure(0.0, 30.0, 18.0)) == 0.0))
    checks.append(("worked example: Pa = 0.5 * (1/3) * 18 * 16 = 48 kN/m", abs(L.active_force(4.0, 30.0, 18.0) - 48.0) < 1e-9))
    # the area of the pressure triangle equals Pa, and its centroid is H/3 above the base
    z = (np.arange(4000) + 0.5) * 4.0 / 4000
    p = L.pressure(z, 30.0, 18.0)
    area = float(np.sum(p) * 4.0 / 4000)
    arm = float(np.sum(p * (4.0 - z)) * 4.0 / 4000) / area
    checks.append(("integrating the pressure gives 48 kN/m acting 1.333 m above the base",
                   abs(area - 48.0) < 1e-3 and abs(arm - 4.0 / 3.0) < 1e-3))
    checks.append(("a 10 kPa surcharge adds Ka q H = 13.33 kN/m", abs(L.surcharge_force(4.0, 30.0, 10.0) - 40.0 / 3.0) < 1e-9))
    pa, ps = L.active_force(4.0, 30.0, 18.0), L.surcharge_force(4.0, 30.0, 10.0)
    arm2 = (pa * 4.0 / 3.0 + ps * 2.0) / (pa + ps)
    checks.append(("with the surcharge the total is 61.33 kN/m and acts 1.478 m above the base",
                   abs(pa + ps - 61.3333) < 1e-3 and abs(arm2 - 1.4783) < 1e-3))
    so, wa, tot = L.saturated_force(4.0, 30.0, 20.0)
    checks.append(("saturated backfill: water alone is 0.5 * 9.81 * 16 = 78.5 kN/m, soil 27.2, total 105.7",
                   abs(wa - 78.48) < 1e-9 and abs(so - 27.1733) < 1e-3 and abs(tot - 105.65) < 0.01))
    checks.append(("saturation raises the force by a factor of about 2.2",
                   abs(tot / L.active_force(4.0, 30.0, 18.0) - 2.2011) < 1e-3))
    checks.append(("phi 25 versus 35 degrees: Ka 0.4059 versus 0.2710, Pa 58.4 versus 39.0 kN/m",
                   abs(L.ka_sin(25) - 0.4059) < 1e-4 and abs(L.ka_sin(35) - 0.2710) < 1e-4
                   and abs(L.active_force(4, 25, 18) - 58.44) < 0.01 and abs(L.active_force(4, 35, 18) - 39.02) < 0.01))
    checks.append(("Ka at 35 degrees is about one third lower than at 25 degrees",
                   abs(1.0 - L.ka_sin(35) / L.ka_sin(25) - 0.3324) < 1e-3))
    checks.append(("a 10 degree error in phi changes the 4 m push by about 50 percent of the lower value (58.44 / 39.02 = 1.498)",
                   abs(L.active_force(4, 25, 18) / L.active_force(4, 35, 18) - 1.4977) < 1e-3))
    checks.append(("the surcharge costs about half a metre of height (4.47 m to 3.95 m)", abs((4.4721 - 3.951) - 0.52) < 0.01))
    checks.append(("hand calculation: H = 3 m, gamma = 19, phi = 30 gives 28.5 kN/m",
                   abs(L.active_force(3.0, 30.0, 19.0) - 28.5) < 1e-9))
    h = L.max_height(60.0, 30.0, 18.0)
    checks.append(("design: the tallest wall with Pa <= 60 kN/m is 4.47 m",
                   abs(h - 4.4721) < 1e-3 and abs(L.active_force(h, 30.0, 18.0) - 60.0) < 1e-9))
    # with a 10 kPa surcharge: 3 H^2 + (10/3) H - 60 = 0
    hs = float(np.max(np.roots([3.0, 10.0 / 3.0, -60.0])))
    checks.append(("design with a 10 kPa surcharge: tallest wall is 3.95 m",
                   abs(hs - 3.951) < 1e-3 and abs(L.active_force(hs, 30.0, 18.0) + L.surcharge_force(hs, 30.0, 10.0) - 60.0) < 1e-6))
    t = L.active_force(4.0, 36.0, 18.0)
    checks.append(("challenge: phi = 36 degrees brings the 4 m push under 40 kN/m but phi = 34 does not",
                   t <= 40.0 and L.active_force(4.0, 34.0, 18.0) > 40.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
